# 00 · Start hier

Welkom bij het IoT-practicum van de les **Klimaatbeheersing in de veehouderij**. Dit notebook controleert of je omgeving werkt en frist de begrippen op die je in de volgende notebooks nodig hebt.

**Hoe werk je met een notebook?** Een notebook bestaat uit tekstcellen (zoals deze) en codecellen. Klik in een codecel en druk op **Shift + Enter** om ze uit te voeren. De uitvoer verschijnt eronder. Voer de cellen **van boven naar beneden** uit.

> Werk je in GitHub Codespaces en vraagt VS Code om een *kernel*? Kies **Python 3.12** (of de hoogste Python-versie die wordt aangeboden).

## 1. Werkt Python en zijn de pakketten geïnstalleerd?

In [ ]:
import sys, platform
import paho.mqtt.client as mqtt
import requests, pandas as pd, matplotlib

print("Python      :", sys.version.split()[0], "op", platform.system())
print("paho-mqtt   :", mqtt.__version__ if hasattr(mqtt, "__version__") else "ok")
print("requests    :", requests.__version__)
print("pandas      :", pd.__version__)
print("matplotlib  :", matplotlib.__version__)
print("\nAlles geladen. Je kunt verder.")

## 2. Begrippen die je nodig hebt

| Begrip | Betekenis in dit practicum |
|---|---|
| **IoT-toestel / node / eindpunt** | Een sensor met een radio en een batterij, die zelfstandig metingen verstuurt. Voorbeeld: Dragino S31-LB. |
| **Verbindingstype** | De *fysieke* weg waarlangs bits reizen: wifi, ethernet, LoRaWAN, Bluetooth, NB-IoT/LTE-M. Vergelijk het met de weg (fietspad, autosnelweg, spoor). |
| **Gegevensuitwisselingsprotocol** | De *afspraak* over hoe berichten eruitzien en wie wanneer spreekt: MQTT, HTTP, CoAP, Modbus. Vergelijk het met de taal en de etiquette die je op die weg gebruikt. |
| **Gateway** | Toestel dat tussen twee verbindingstypes vertaalt, bv. van LoRaWAN naar ethernet/internet. |
| **Broker** | De centrale "postbus" van MQTT: toestellen publiceren berichten op een *topic*, andere toestellen abonneren zich op dat topic. |
| **Topic** | Een adres met schuine strepen, bv. `stal1/afdeling3/temperatuur`. |
| **Payload** | De eigenlijke inhoud van een bericht: ruwe bytes (hex), tekst of JSON. |
| **JSON** | Tekstformaat met sleutels en waarden: `{"temperature": 21.4, "humidity": 63.2}`. |
| **Uplink / downlink** | Bericht van sensor naar server / van server naar sensor. |

Een vaste **rode draad** doorheen het practicum: een temperatuur- en vochtsensor in een stalafdeling verstuurt elke 10 minuten een meting, die uiteindelijk op het scherm van de veehouder of in de klimaatcomputer belandt.

## 3. Verbinding met de MQTT-broker testen

We gebruiken de **publieke broker van HiveMQ** (`broker.hivemq.com`). Iedereen ter wereld mag ermee verbinden, dus gebruik ze nooit voor echte of persoonlijke gegevens. Onderstaande cel verbindt, abonneert zich op een testtopic, publiceert één bericht en wacht tot ze dat eigen bericht terugkrijgt.

In [ ]:
import paho.mqtt.client as mqtt
import time, uuid

BROKER = "broker.hivemq.com"
POORT  = 1883                       # standaard MQTT-poort (onversleuteld)
TOPIC  = f"ugent/i700301/test/{uuid.uuid4().hex[:8]}"   # uniek testtopic

ontvangen = []

def bij_bericht(client, userdata, msg):
    ontvangen.append(msg.payload.decode())

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_message = bij_bericht
client.connect(BROKER, POORT, keepalive=60)
client.loop_start()
client.subscribe(TOPIC)
time.sleep(1)
client.publish(TOPIC, "hallo vanuit mijn notebook")

for _ in range(50):                  # max. 5 s wachten
    if ontvangen:
        break
    time.sleep(0.1)

client.loop_stop(); client.disconnect()
print("Topic   :", TOPIC)
print("Ontvangen:", ontvangen or "NIETS — zie de tip hieronder")

**Lukt het niet (geen bericht of een time-out)?** Sommige netwerken blokkeren poort 1883. Gebruik dan MQTT **over websockets** op poort 8884 met TLS; dat lijkt voor het netwerk op gewoon https-verkeer. Vervang in de cel hierboven de regel `client = mqtt.Client(...)` en `client.connect(...)` door:

```python
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, transport="websockets")
client.tls_set()
client.ws_set_options(path="/mqtt")
client.connect("broker.hivemq.com", 8884, keepalive=60)
```

In GitHub Codespaces werkt poort 1883 normaal gezien gewoon.

## 4. Verbinding met een HTTP-API testen

HTTP is het protocol van het web. Een *API* is een webadres dat geen webpagina maar gegevens (meestal JSON) teruggeeft. Onderstaande cel vraagt bij Open-Meteo alleen de actuele temperatuur in Gent. In de opdracht eronder pas je diezelfde oproep zelf aan.

In [ ]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
parameters = {
    "latitude": 51.05, "longitude": 3.72,          # Gent
    "current": "temperature_2m",
}
antwoord = requests.get(url, params=parameters, timeout=10)
print("HTTP-statuscode:", antwoord.status_code, "(200 = OK)")

gegevens = antwoord.json()                          # JSON-tekst -> Python-dictionary
huidig = gegevens["current"]
print(f"Gent, {huidig['time']} UTC: {huidig['temperature_2m']} °C")

### Opdracht. Vraag zelf ander weer op

De cel hierboven haalt alleen de temperatuur in Gent op. Kopieer ze naar de lege cel hieronder en pas ze aan. De namen die je daarvoor nodig hebt, staan niet in dit notebook. Die zie je op de documentatiepagina.

Open <https://open-meteo.com/en/docs>. Bovenaan vul je een breedte- en lengtegraad in, bij Latitude en Longitude. Voor Gent zijn dat `51.05` en `3.72`, dezelfde getallen als hierboven. Daaronder vink je aan wat je wilt ophalen. **Current Weather** is het weer van nu. **Hourly Weather Variables** is een reeks, uur per uur; die heb je pas in de laatste stap nodig. Scroll daarna naar **API Response** en klik op **Python**. Daar staat de code die bij jouw vinkjes hoort. Neem die namen over in `parameters`. De voorbeeldstad van de website laat je links liggen.

Je moet geen sensor opzoeken. Een gewone coördinaat is genoeg, en de API kiest zelf het dichtstbijzijnde punt van haar weermodel. Dat punt krijg je terug als `latitude` en `longitude`. Het ligt soms een paar kilometer naast wat je vroeg, dus druk allebei af. Wil je twee dingen tegelijk, zoals temperatuur en vochtigheid, dan zet je ze samen in `current`, met een komma ertussen. De eenheid staat mee in het antwoord (`current_units` of `hourly_units`). Die hoef je niet uit het hoofd te kennen.

1. **Ook de vochtigheid in Gent.** Vink onder Current Weather *Relative Humidity (2 m)* aan. Zet de naam uit het Python-voorbeeld naast `temperature_2m`, en druk temperatuur en RV af.
2. **Een plek die je zelf kiest.** Thuis, de stal, of ergens anders. Rechtsklik in Google Maps op die plek. De twee getallen bovenaan het menu zijn de breedte en de lengte. Zet die in `latitude` en `longitude` en vraag alleen de temperatuur. Druk af waar je om vroeg, en welk punt de API echt gebruikte.
3. **Alles van nu, op diezelfde plek.** Vink onder Current Weather alle vakjes aan en zet die namen in `current`. Druk elke waarde af, met de eenheid uit het antwoord.
4. **Die plek over meerdere dagen.** Vink temperatuur en vochtigheid aan onder Hourly Weather Variables. Zet bovenaan Past days op minstens 1 en Forecast days op meer dan 1. In de parametertabel op dezelfde pagina heet de tijdzone `timezone`; zet die op `Europe/Brussels`. De cijfers zitten nu in `hourly`, niet meer in `current`. Druk hoeveel uren je kreeg, en de temperatuur en RV van het eerste en het laatste uur.

De ingeklapte cel hieronder doet stap 3 en 4 voor Kortrijk. Dat is een voorbeeld. Jij neemt een plek die je zelf kiest.

In [ ]:
# Plak hier eerst de temperatuurcel, en pas ze per stap aan.
# De namen staan in het Python-voorbeeld op https://open-meteo.com/en/docs

# Stap 1. Gent. current = temperatuur en relatieve vochtigheid.

# Stap 2. Een plek naar keuze. Andere latitude en longitude. Alleen de temperatuur.
#         Druk je coördinaten én gegevens["latitude"], gegevens["longitude"].

# Stap 3. Dezelfde plek. current = alle vinkjes onder Current Weather.

# Stap 4. Dezelfde plek. hourly, past_days, forecast_days, timezone.
#         De reeks zit in gegevens["hourly"]. Eerste en laatste uur.


In [ ]:
# Voorbeeld — Kortrijk. Zelf vervang je de coördinaten door die van je eigen plek.
import requests

url = "https://api.open-meteo.com/v1/forecast"
gevraagd_lat, gevraagd_lon = 50.828, 3.265

parameters = {
    "latitude": gevraagd_lat,
    "longitude": gevraagd_lon,
    "current": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature",
        "is_day",
        "snowfall",
        "weather_code",
        "cloud_cover",
        "pressure_msl",
        "surface_pressure",
        "wind_speed_10m",
        "wind_direction_10m",
        "wind_gusts_10m",
    ]),
    "cell_selection": "nearest",
}
antwoord = requests.get(url, params=parameters, timeout=10)
antwoord.raise_for_status()
gegevens = antwoord.json()
huidig, eenheden = gegevens["current"], gegevens["current_units"]

print(f"Gevraagd:    {gevraagd_lat}, {gevraagd_lon}")
print(f"Roosterpunt: {gegevens['latitude']}, {gegevens['longitude']}")
print(huidig["time"])
for sleutel, waarde in huidig.items():
    if sleutel in ("time", "interval"):
        continue
    print(f"{sleutel}: {waarde} {eenheden[sleutel]}")

reeks = requests.get(url, params={
    "latitude": gevraagd_lat,
    "longitude": gevraagd_lon,
    "hourly": "temperature_2m,relative_humidity_2m",
    "past_days": 1,
    "forecast_days": 2,
    "timezone": "Europe/Brussels",
    "cell_selection": "nearest",
}, timeout=10)
reeks.raise_for_status()
uurlijks = reeks.json()["hourly"]
eenheid = reeks.json()["hourly_units"]
tijden = uurlijks["time"]
print(f"{len(tijden)} tijdstappen, {tijden[0]} → {tijden[-1]}")
for i in (0, -1):
    print(f"{tijden[i]}: {uurlijks['temperature_2m'][i]} {eenheid['temperature_2m']}, "
          f"{uurlijks['relative_humidity_2m'][i]} {eenheid['relative_humidity_2m']}")


## 5. Klaar

Werkten de MQTT-test en de temperatuur-oproep, en heb je de opdracht hierboven gemaakt? Ga dan naar `01_sensor_kiezen_en_verbinden.ipynb`.

Onthoud het verschil dat je net zag:

* Bij **MQTT** maak je één verbinding, en daarna *duwt* de broker berichten naar jou zodra ze er zijn (publish/subscribe).
* Bij **HTTP** *vraag* je telkens expliciet om gegevens en krijg je één antwoord (request/response). Welke gegevens je krijgt, stuur je met de parameters in de URL. Die namen staan in de documentatie van de API, niet in de voorbeeldcode.